# How the primary outcome is derived

**Question.** In late-window, CT-perfusion-selected stroke, does intravenous thrombolysis before
thrombectomy (*bridging*) improve 90-day functional outcome compared with thrombectomy alone (*EVT alone*)?

**Primary outcome.** The 90-day modified Rankin Scale (mRS), an ordinal score from 0 (no symptoms) to 6 (death).

**Primary quantity.** One number, the *common odds ratio* of a better mRS under bridging, estimated in the
population where either treatment was a realistic option. Values > 1 favour bridging.

This notebook walks the chain that produces it:

```
workbook ──► cohort ──► propensity score ──► overlap weights ──► weighted mRS contrast ──► bootstrap
   §1          §1            §3                   §4                   §5, §6                §7
```

Every step calls the analysis code itself (`extended_bridging/*.py`); nothing is re-implemented here.
Section references `[§n]` point to `statistical_analysis_plan.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The analysis modules live one directory up.
sys.path.insert(0, str(Path.cwd().parent))

import config as C
import data, derive, eligibility, cohort, propensity, outcome, bootstrap, model

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

ARM = C.TREATMENT_LABELS        # {0: "EVT alone", 1: "bridging"}
COLOURS = {0: "#7f8c8d", 1: "#2980b9"}

## 1. From workbook to cohort

Four cleaning and selection steps, in the order the pipeline runs them. Each appends its reasoning to an
audit log; here we only keep the counts.

In [ ]:
df, audit = data.load(data.WORKBOOK)          # read, rename, type-check, hash-verify
n_source = len(df)

df = derive.derive(df, audit)                 # onset type, mRS dichotomies (never used here)
df = eligibility.classify(df, audit)          # who could have received IVT at all
df = cohort.build(df, audit)                  # keep IVT-eligible patients at centres using both strategies

print(f"records in workbook : {n_source}")
print(f"analysis cohort     : {len(df)}")
print(df[C.TREATMENT].map(ARM).value_counts().rename("patients per arm"))

### The raw outcome, before any adjustment

Each bar is one arm; segments are the share of patients at each mRS level. Lighter = better.

In [ ]:
def mrs_shares(y, w=None):
    # Share of (weighted) patients at each mRS level 0..6.
    w = np.ones(len(y)) if w is None else w
    return np.array([w[y == k].sum() for k in C.MRS_LEVELS]) / w.sum()


def stacked_mrs(ax, shares_by_arm, title):
    # Horizontal stacked bars, one per arm, mRS 0 on the left.
    cmap = plt.get_cmap("RdYlGn_r", len(C.MRS_LEVELS))
    for i, (code, shares) in enumerate(shares_by_arm.items()):
        left = 0.0
        for k, s in zip(C.MRS_LEVELS, shares):
            ax.barh(i, s, left=left, color=cmap(k), edgecolor="white")
            if s > 0.04:
                ax.text(left + s / 2, i, str(k), ha="center", va="center", fontsize=8)
            left += s
    ax.set_yticks(range(len(shares_by_arm)), [ARM[c] for c in shares_by_arm])
    ax.set_xlim(0, 1); ax.set_xlabel("share of patients"); ax.set_title(title)


y_all = df[C.PRIMARY_OUTCOME].to_numpy(dtype=float)
a_all = df[C.TREATMENT].to_numpy(dtype=float)
present = ~np.isnan(y_all)

fig, ax = plt.subplots(figsize=(8, 2.2))
stacked_mrs(ax, {c: mrs_shares(y_all[present & (a_all == c)]) for c in ARM}, "90-day mRS, unweighted")
plt.show()

## 2. Why the raw comparison is not enough

Treatment was not randomised. Clinicians gave IVT to some patients and not others for reasons that
also predict outcome (age, stroke severity, infarct core, ...). A raw difference between arms mixes
the treatment effect with those differences. Below, the arms are compared on the prespecified confounders `[§6]`.

In [ ]:
numeric = [c for c in C.PS_COVARIATES if pd.api.types.is_numeric_dtype(df[c])]
baseline = df.groupby(df[C.TREATMENT].map(ARM))[numeric].mean().T
baseline.columns.name = None
baseline

## 3. The propensity score

The *propensity score* `e(X)` is each patient's estimated probability of receiving bridging, given the
confounders. It is fitted with a Firth-penalised logistic regression `[§7]`: the penalty keeps the fit
stable in a small sample where ordinary logistic regression can fail.

Two patients with the same `e(X)` are, on the measured confounders, equally likely to have been treated,
so a difference in their outcomes cannot be explained by those confounders.

In [ ]:
ps = propensity.fit(df, audit)

coef = pd.Series(ps.fit.beta[1:], index=ps.fit.columns, name="log-odds of bridging")
display(coef.to_frame())

fig, ax = plt.subplots(figsize=(7, 3))
bins = np.linspace(0, 1, 21)
for c in ARM:
    ax.hist(ps.e[ps.in_model & (df[C.TREATMENT] == c)], bins=bins, alpha=0.6,
            color=COLOURS[c], label=ARM[c])
ax.set_xlabel("propensity score e(X)"); ax.set_ylabel("patients"); ax.legend()
ax.set_title("Where do the two arms overlap?")
plt.show()

The histogram shows the key issue: some patients were near-certain to get one treatment (`e` close to
0 or 1). For them the other arm has almost no comparable patient, and any estimate there would rest on
extrapolation.

## 4. Overlap weights

Rather than trying to estimate the effect in *everyone*, the plan targets the patients for whom either
choice was realistic — the *overlap population* (estimand ATO, `[§7]`). This is done with weights:

```
bridging  patient :  w = 1 − e(X)
EVT alone patient :  w = e(X)
```

A treated patient who was very likely to be treated (`e ≈ 1`) tells us little about what would have
happened otherwise, so gets weight ≈ 0. A patient near `e = 0.5` gets the largest weight in both arms.
Weights are bounded between 0 and 1, so no single patient can dominate.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for c in ARM:
    m = ps.in_model & (df[C.TREATMENT] == c)
    ax.scatter(ps.e[m], ps.w[m], s=18, color=COLOURS[c], label=ARM[c])
ax.set_xlabel("propensity score e(X)"); ax.set_ylabel("overlap weight w"); ax.legend()
ax.set_title("Weight as a function of the propensity score")
plt.show()

# Effective sample size: how many equally weighted patients the weighted arm is worth.
pd.DataFrame({"patients": df.loc[ps.in_model, C.TREATMENT].map(ARM).value_counts(),
              "effective sample size": {ARM[c]: v for c, v in ps.ess.items()}})

## 5. The weighted outcome contrast

The weights are applied to the mRS distributions. Everything from here on is computed on the
*estimation population*: patients with a propensity score **and** a recorded 90-day mRS `[§11]`.

In [ ]:
in_est = outcome.estimation_population(df, ps)
sub = df.loc[in_est]

y = sub[C.PRIMARY_OUTCOME].to_numpy(dtype=float)
a = sub[C.TREATMENT].to_numpy(dtype=float)
w = ps.w.loc[in_est].to_numpy(dtype=float)

print(f"estimation population: {in_est.sum()} of {len(df)} cohort patients")

fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True)
stacked_mrs(axes[0], {c: mrs_shares(y[a == c]) for c in ARM}, "unweighted")
stacked_mrs(axes[1], {c: mrs_shares(y[a == c], w[a == c]) for c in ARM}, "overlap-weighted")
plt.tight_layout(); plt.show()

### Cumulative risk differences `RD_k`

An mRS shift is easiest to read on the absolute scale. For each cut `k = 0…5`:

```
RD_k = P(mRS ≤ k | bridging) − P(mRS ≤ k | EVT alone)        (weighted)
```

Positive `RD_k` means more bridging patients reach mRS ≤ k. These six numbers *describe* the contrast;
they are reported with confidence intervals but **no p-values**, because testing six cuts of one
contrast invites picking the most favourable one `[§8]`.

In [ ]:
rd, cumulative = outcome.cumulative_rd(y, a, w)

rd_table = pd.DataFrame({
    f"P(mRS ≤ k), {ARM[0]}": {k: cumulative[k][0] for k in C.MRS_THRESHOLDS},
    f"P(mRS ≤ k), {ARM[1]}": {k: cumulative[k][1] for k in C.MRS_THRESHOLDS},
    "RD_k": rd,
})
rd_table.index.name = "k"
rd_table

## 6. The primary quantity: a common odds ratio

To summarise the whole shift with **one** number, the plan fits a weighted *proportional-odds* model
with treatment `A` as the only predictor `[§8]`:

```
logit P(mRS ≤ k) = α_k + β · A          for every cut k
```

- `α_k` — one intercept per cut; these reproduce the EVT-alone distribution.
- `β` — a **single** shift shared by all cuts. That is the "proportional odds" assumption.
- `exp(β)` — the *common odds ratio*: the odds of being at or below any given mRS level (i.e. doing at
  least that well) under bridging, relative to EVT alone. `> 1` favours bridging.

Why a model when we already have the six `RD_k`? Because a study this small cannot estimate six
independent effects with any precision; one shared parameter borrows strength across the whole scale.

In [ ]:
est = outcome.primary(df, ps, audit)

fit_table = pd.DataFrame({
    "parameter": [f"α at mRS ≤ {k}" for k in est.cut_levels] + ["β (bridging)", "exp(β)"],
    "estimate": list(est.alpha) + [est.beta, est.odds_ratio],
}).set_index("parameter")
fit_table

### Does one shared `β` describe the data?

Compare the weighted empirical cumulative probabilities (§5) with those the fitted model implies.
Close agreement means the proportional-odds summary is a fair description.

In [ ]:
def model_cumulative(arm):
    # P(mRS ≤ k | arm) implied by the fit: inverse-logit of α_k + β·A.
    return 1 / (1 + np.exp(-(np.asarray(est.alpha) + est.beta * arm)))


fig, ax = plt.subplots(figsize=(7, 3.5))
ks = list(C.MRS_THRESHOLDS)
for c in ARM:
    ax.plot(ks, [cumulative[k][c] for k in ks], "o", color=COLOURS[c], label=f"{ARM[c]}, weighted data")
    ax.plot(est.cut_levels, model_cumulative(c), "-", color=COLOURS[c], alpha=0.7, label=f"{ARM[c]}, model")
ax.set_xlabel("k"); ax.set_ylabel("P(mRS ≤ k)"); ax.legend(fontsize=8)
ax.set_title("Weighted data vs proportional-odds fit")
plt.show()

## 7. Uncertainty: the bootstrap

The model gives no usable standard error: the weights come from an *estimated* propensity score, and a
model-based error would pretend they were known. Instead `[§10]`:

1. Resample patients with replacement, **within each centre**, to make a new dataset of the same size.
2. Re-run steps 3–6 on it: refit the propensity model, recompute the weights, refit the ordinal model.
3. Repeat 2000 times, with a recorded seed. Replicates whose fit fails are dropped and counted, never patched.

The spread of the 2000 `β` values is the sampling uncertainty of the estimate.

- **95% CI** — the 2.5th and 97.5th percentiles of the draws.
- **p-value** — the share of draws on the far side of `β = 0`, doubled: `p = 2·min{Pr(β* ≤ 0), Pr(β* ≥ 0)}`.
  It is the smallest level at which the interval excludes "no effect", so CI and p always agree.

The primary outcome has exactly **one** test, on `β`. The six `RD_k` receive intervals only.

In [ ]:
scratch = data.Audit(data.WORKBOOK)      # replicate logs are discarded


def one_replicate(draw):
    # Steps 3–6 on a resampled cohort. A failed fit returns None ("dropped and counted").
    try:
        ps_b = propensity.fit(draw, scratch)
        est_b = outcome.primary(draw, ps_b, scratch)
    except model.FitError:
        return None
    return est_b.beta, tuple(est_b.rd[k] for k in C.MRS_THRESHOLDS)


results = bootstrap.replicates(df, one_replicate, n=C.N_BOOT, seed=C.SEED, stratum=C.BOOT_STRATUM)
kept = [r for r in results if r is not None]

beta_draws = np.array([r[0] for r in kept])
rd_draws = np.array([r[1] for r in kept])
print(f"replicates attempted {C.N_BOOT}, kept {len(kept)}, dropped {C.N_BOOT - len(kept)}")

In [ ]:
lo, hi = bootstrap.percentile_ci(beta_draws)
p = bootstrap.bootstrap_p(beta_draws)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(beta_draws, bins=40, color="#95a5a6")
ax.axvline(est.beta, color="black", label=f"estimate β = {est.beta:.3f}")
ax.axvspan(lo, hi, color="#2980b9", alpha=0.15, label="95% percentile CI")
ax.axvline(0, color="red", ls="--", label="null: β = 0")
ax.set_xlabel("β in bootstrap replicate"); ax.set_ylabel("replicates"); ax.legend(fontsize=8)
plt.show()

print(f"common odds ratio {est.odds_ratio:.2f}  (95% CI {np.exp(lo):.2f} to {np.exp(hi):.2f}),  p = {p:.3f}")

In [ ]:
# Intervals for the absolute-scale presentation; no p-values by design [§8].
rd_ci = pd.DataFrame(
    [(rd[k], *bootstrap.percentile_ci(rd_draws[:, i])) for i, k in enumerate(C.MRS_THRESHOLDS)],
    index=pd.Index(C.MRS_THRESHOLDS, name="k"), columns=["RD_k", "CI lower", "CI upper"])
rd_ci

## 8. Reading the result

- **What was estimated.** The common odds ratio of a better 90-day mRS with bridging versus EVT alone,
  among IVT-eligible patients at both-arm centres who could plausibly have received either treatment.
- **How.** Confounders enter only through the propensity score; the outcome model contains treatment
  alone. The estimate is therefore marginal in the overlap population, not adjusted for covariates.
- **One number hides shape.** The proportional-odds assumption is not tested `[§16a]`. When the six
  `RD_k` do not share a sign (bridging ahead at some cuts, behind at others), an odds ratio near 1 means
  "no *uniform* shift", not "no difference anywhere" — read it beside the `RD_k` table.
- **Uncertainty.** From a bootstrap that repeats *every* estimation step, so the interval includes the
  cost of estimating the weights.
- **Caveats stated in the plan.** The overlap population is defined by the propensity model, so a
  different model targets a different population `[§7]`. Confounders absent from the data (ASPECTS,
  occlusion site, collaterals, onset-to-presentation time) cannot be adjusted for; an E-value is
  reported elsewhere `[§13]`. Entry requires thrombectomy, which is downstream of the IVT decision `[§2]`.